# GloVe (tf-idf weighted mean) + shared bidirectional LSTM over two candidate stories + MLP

**Task:** Abductive Commonsense Reasoning (ART). Given two observations O1, O2 and two
hypotheses H1, H2, pick the hypothesis that better explains what happened in between.

**Approach of this notebook:** Same as `glove_lstm_bidirectional.ipynb`, only the input framing and the classifier input change
(details in the model card `models/project_1/glove_lstm_bidirectional_improved/README.md`):

- same preprocessing, GloVe sentence vectors (`glove-wiki-gigaword-300`) as tf-idf weighted mean, the same weighting as in
  `static/final_presentable_notebook/glove_tfidf_mlp.ipynb`: rare words count more. The idf weights are fixed numbers, not learned
- candidate framing: every example becomes two stories, A = O1 -> H1 -> O2 and B = O1 -> H2 -> O2, 3 time steps of 300 dims each.
  The LSTM reads each hypothesis between the two observations, so it can model whether the hypothesis bridges them
- the same bidirectional LSTM (shared weights) encodes A and B: h_A and h_B, each `[h_forward ; h_backward]` = 2 * 256 dims
- the classifier gets both stories side by side: `[h_A ; h_B]` = 2 * 512 dims
- MLP head: linear -> ReLU -> linear -> 2 logits, softmax -> probability for H1 and H2
- dropout 0.3 in the classifier head (dropout -> linear -> relu -> dropout -> linear), otherwise hyperparameters, training and logging unchanged

In [1]:
%pip install -q datasets gensim lightning nltk python-dotenv wandb


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))  # notebooks/project_1 -> shared pipeline package (this notebook is one folder deeper)

import lightning as L
import torch
import torch.nn as nn

from pipeline import tracking
from pipeline.data import DATASET, load_splits
from pipeline.evaluation import predict_hypotheses
from pipeline.features.static_embeddings import MeanEmbedder, TfidfMeanEmbedder, build_features, combine_concat_candidate_framing, load_vectors
from pipeline.features.tfidf import fit_vectorizer, idf_weights
from pipeline.loaders import make_loader
from pipeline.preprocessing import tokenize_split
from pipeline.preprocessing import nltk_alnum_lower as preprocessing  # swap the preprocessing variant here
from pipeline.training import evaluate_best, fit, training_summary, weights_dir

In [3]:
tracking.login()

environment loaded from: /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/.env


wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: cmedi32 (cmedi32-hochschule-luzern) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
config = {
    "dataset": DATASET,
    **preprocessing.CONFIG,  # tokenizer, lowercase, remove_punctuation, remove_stopwords, normalization, truncation
    "embedding_dim": 300,  # dimensions per sentence vector = LSTM input per time step, glove-wiki-gigaword comes with 50, 100, 200 or 300
    "sentence_representation": "weighted_mean",
    "sentence_weighting": "tfidf",  # "tfidf" or "none" (plain mean, for the ablation)
    "tfidf_fit_on": "train sentences (O1, O2, H1, H2)",
    "example_representation": "candidate_framing",  # A = O1, H1, O2 and B = O1, H2, O2 -> 2 sequences of 3 rnn time steps
    "shared_encoder": True,  # the same LSTM encodes A and B
    "encoder": "lstm",  # rnn, gru, lstm
    "bidirectional": True,
    "num_layers": 1,  # stacked LSTM layers, dropout between them when > 1
    "rnn_hidden_dim": 256,  # LSTM state per direction: h_A and h_B get 2 * rnn_hidden_dim when bidirectional
    "classifier": "mlp",  # dropout -> linear -> relu -> dropout -> linear
    "dropout": 0.3,  # share of classifier inputs (and of the outputs between stacked LSTM layers) zeroed during training, 0.0 = off, chosen by loss/valid (series above)
    "hidden_dim": 256,  # hidden layer of the classifier head: linear([h_A ; h_B], hidden_dim) -> relu -> linear(hidden_dim, 2)
    "learning_rate": 1e-3,
    "weight_decay": 0.0,  # AdamW: every step shrinks all weights a little towards 0, 0.0 = off (plain Adam), e.g. 1e-4 to 1e-2
    "gradient_clip_val": 0.0,  # max length (L2 norm) of all gradients together per step, longer ones are scaled down, 0.0 = off, e.g. 1.0
    "batch_size": 64,
    "max_epochs": 100,
    "patience": 5,  # validation checks without improvement of loss/valid before stopping
    "seed": 42,
}
config["embedding"] = f"glove-wiki-gigaword-{config['embedding_dim']}"  # gensim model id, e.g. "glove-wiki-gigaword-300"
# the shapes in the code comments of this notebook are for the defaults above (300 dims per sentence, rnn_hidden_dim 256, hidden_dim 256)

# one name per variant, so the ablation does not overwrite the checkpoint of the main run
# W&B run + artifact name, weights in models/project_1/<name with _>/weights
RUN_NAME = {"tfidf": "glove-lstm-bidirectional-improved", "none": "glove-lstm-bidirectional-improved-mean"}[config["sentence_weighting"]]

run = tracking.init_run(RUN_NAME, config)

In [5]:
train, valid, test = load_splits()

train: 16801, valid: 1000, test: 1532


In [6]:
train_tokens = tokenize_split(train, preprocessing.preprocess)
valid_tokens = tokenize_split(valid, preprocessing.preprocess)
test_tokens = tokenize_split(test, preprocessing.preprocess)

Map:   0%|          | 0/16801 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1532 [00:00<?, ? examples/s]

In [7]:
wv = load_vectors(config["embedding"])

vocabulary: 400000 words, 300 dimensions


### TF-IDF weighting

Same weighting as in `glove_tfidf_mlp.ipynb`: every word vector is weighted by the idf of the word before averaging a sentence
(`pipeline/features/tfidf.py: idf_weights`, `pipeline/features/static_embeddings.py: TfidfMeanEmbedder`).

In [8]:
# documents for the idf: every single training sentence (O1, O2, H1, H2), already tokenized
vectorizer = fit_vectorizer(train_tokens)
idf, idf_unseen = idf_weights(vectorizer, train_tokens)  # dict word -> idf, idf for words never seen in training

vocabulary: 16189 terms
tf-idf fitted on 67204 training sentences, vocabulary: 16189 terms
idf range: 2.06 – 11.42, words never seen in train: 12.12


In [9]:
if config["sentence_weighting"] == "tfidf":
    embed = TfidfMeanEmbedder(wv, idf, idf_unseen)  # rare words count more, fixed weights: not learned
else:
    embed = MeanEmbedder(wv)  # plain mean, for the ablation

In [10]:
# one row per example: story A (O1, H1, O2) followed by story B (O1, H2, O2) -> 6 * 300 = 1800 dims
X_train, y_train = build_features(train_tokens, embed, combine_concat_candidate_framing)
X_valid, y_valid = build_features(valid_tokens, embed, combine_concat_candidate_framing)
X_test, y_test = build_features(test_tokens, embed, combine_concat_candidate_framing)

print(f"input: {X_train.shape[1]} dimensions, {X_train.dtype}")
print(embed.summary())  # oov counters (+ tf-idf counters for the weighted mean)

input: 1800 dimensions, float32
{'oov/tokens': 670395, 'oov/occurrences': 1144, 'oov/rate': 0.0017064566412338994, 'oov/unique_types': 850, 'oov/empty_sentences': 7, 'tfidf/vocabulary_size': 16189, 'tfidf/unseen_tokens': 1417}


### DataLoaders - Takes care of Training Batching and Loading

In [11]:
L.seed_everything(config["seed"])

train_loader = make_loader(X_train, y_train, config["batch_size"], shuffle=True)
valid_loader = make_loader(X_valid, y_valid, config["batch_size"], shuffle=False)
test_loader = make_loader(X_test, y_test, config["batch_size"], shuffle=False)

Seed set to 42


### LightningModule Model

In [12]:
class CandidateLSTMClassifier(L.LightningModule):

    def __init__(self, input_dim, rnn_hidden_dim, hidden_dim, output_dim, num_layers, bidirectional, dropout, learning_rate, weight_decay):
        super().__init__()
        self.save_hyperparameters()
        inter_layer_dropout = dropout if num_layers > 1 else 0.0  # only acts between stacked layers, pytorch warns for 1 layer
        self.rnn = nn.LSTM(input_dim, rnn_hidden_dim, num_layers=num_layers, bidirectional=bidirectional, dropout=inter_layer_dropout, batch_first=True)  # one encoder for story A and story B
        num_directions = 2 if bidirectional else 1
        encoding_dim = rnn_hidden_dim * num_directions  # size of h_A and of h_B: 2 * 256 = 512
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(2 * encoding_dim, hidden_dim),  # [h_A ; h_B] = 2 * 512 = 1024 classidier takes both concatenations
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
        )
        self.loss_fn = nn.CrossEntropyLoss()

    def encode(self, story):
        # story: (batch, 3, 300), one time step per sentence
        _, (hidden, _) = self.rnn(story)  # hidden h: (num_layers * num_directions, batch, rnn_hidden_dim), cell state c unused
        if self.hparams.bidirectional:
            return torch.cat([hidden[-2], hidden[-1]], dim=1)  # last layer: forward state after O2, backward state after O1
        return hidden[-1]  # last layer: state after reading O2

    def forward(self, x):
        # (batch, 6 * 300) -> (batch, 2, 3, 300): two stories with 3 time steps each
        stories = x.view(x.size(0), 2, -1, self.hparams.input_dim)
        h_a = self.encode(stories[:, 0])  # O1 -> H1 -> O2
        h_b = self.encode(stories[:, 1])  # O1 -> H2 -> O2, same LSTM = shared weights
        features = torch.cat([h_a, h_b], dim=1)  # both stories side by side: (batch, 1024)
        return self.classifier(features)  # logits: (batch, 2)

    def configure_optimizers(self):
        # AdamW = Adam with decoupled weight decay: each step multiplies every weight by (1 - learning_rate * weight_decay).
        # Adam's own weight_decay is added to the gradient instead, where the adaptive scaling weakens it. 0.0 = plain Adam
        return torch.optim.AdamW(self.parameters(), lr=self.hparams.learning_rate, weight_decay=self.hparams.weight_decay)

    def shared_step(self, batch, split):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)  # CrossEntropyLoss applies the softmax itself, so it gets the raw logits
        accuracy = (logits.argmax(dim=1) == y).float().mean()
        self.log(f"loss/{split}", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log(f"accuracy/{split}", accuracy, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self.shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self.shared_step(batch, "valid")

    def test_step(self, batch, batch_idx):
        return self.shared_step(batch, "test")

    def predict_step(self, batch, batch_idx):
        x, _ = batch
        return torch.softmax(self(x), dim=1)  # (batch, 2): p(hypothesis_1), p(hypothesis_2), each row sums to 1

In [13]:
model = CandidateLSTMClassifier(
    input_dim=wv.vector_size,  # one time step = one 300 dim sentence vector
    rnn_hidden_dim=config["rnn_hidden_dim"],
    hidden_dim=config["hidden_dim"],
    output_dim=2,
    num_layers=config["num_layers"],
    bidirectional=config["bidirectional"],
    dropout=config["dropout"],
    learning_rate=config["learning_rate"],
    weight_decay=config["weight_decay"],
)

trainer = fit(
    model, train_loader, valid_loader, weights_dir(RUN_NAME),
    max_epochs=config["max_epochs"], patience=config["patience"], gradient_clip_val=config["gradient_clip_val"],
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/loggers/wandb.py:400: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_lstm_bidirectional_improved/weights exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ rnn        │ LSTM             │  1.1 M │ train │     0 │
│ 1 │ classifier │ Sequential       │  262 K │ train │     0 │
│ 2 │ loss_fn    │ CrossEntropyLoss │      0 │ train │     0 │
└───┴────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 1.4 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.4 M                                                                                                
Total estimated model params size (MB): 5.623                                                                      
Modules in train mode: 8                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, 
LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 
'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.

Metric loss/valid improved. New best score: 0.657
Metric loss/valid improved by 0.017 >= min_delta = 0.0. New best score: 0.640
Metric loss/valid improved by 0.033 >= min_delta = 0.0. New best score: 0.608
Metric loss/valid improved by 0.024 >= min_delta = 0.0. New best score: 0.584
Monitored metric loss/valid did not improve in the last 5 records. Best score: 0.584. Signaling Trainer to stop.


### Best Checkpoint + Upload Artifacts to Weights and Biases

In [14]:
evaluate_best(trainer, model, valid_loader, test_loader)

run.summary.update({
    **training_summary(trainer, y_test),  # accuracy/majority, best/loss_valid, best/checkpoint, early_stopping/stopped_epoch
    **embed.summary(),  # oov/..., for tf-idf also tfidf/vocabulary_size, tfidf/unseen_tokens
})

Restoring states from the checkpoint path at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_lstm_bidirectional_improved/weights/best-epoch=3-v1.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_lstm_bidirectional_improved/weights/best-epoch=3-v1.ckpt


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      accuracy/valid       │    0.6980000138282776     │
│        loss/valid         │    0.5841435790061951     │
└───────────────────────────┴───────────────────────────┘

Restoring states from the checkpoint path at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_lstm_bidirectional_improved/weights/best-epoch=3-v1.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_lstm_bidirectional_improved/weights/best-epoch=3-v1.ckpt


Output()

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       accuracy/test       │    0.5182767510414124     │
│         loss/test         │    0.8854434490203857     │
└───────────────────────────┴───────────────────────────┘

In [15]:
tracking.log_best_checkpoint(run, RUN_NAME, trainer.checkpoint_callback)

### Probability per Hypothesis and Answer

In [16]:
results = predict_hypotheses(trainer, model, test_loader, test)
results.head(10)

Restoring states from the checkpoint path at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_lstm_bidirectional_improved/weights/best-epoch=3-v1.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_lstm_bidirectional_improved/weights/best-epoch=3-v1.ckpt


Output()

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


accuracy from the answers: 0.5183


,observation_1,observation_2,hypothesis_1,hypothesis_2,label,p_hypothesis_1,p_hypothesis_2,answer
0,A man drove while listening to the radio.,The man called the radio station to ask.,The man heard a song he liked but the radio st...,He wanted to know the name of a song.,2,0.368534,0.631466,2
1,Ada was emptying the trash one night.,Ada had to pick up all the trash from the floor.,the bag broke as Ada was walking out the door.,The trash bag ripped open from a hole in the top.,1,0.409985,0.590015,2
2,Alex was home alone on a Friday night.,"Three friends were there to visit him, and the...",Alex was sad to spend the whole evening alone.,"Alex,posted on facebook that he was lonely at ...",2,0.109370,0.890630,2
3,Amber always thought she was the best tennis p...,Amber was disappointed that she lost to a new ...,amber is tennis player so good for playing but...,She be saded against a new player.,1,0.102594,0.897406,2
4,Amy decided to move from Wisconsin to Florida.,"However, she discovered Alaska was a lot of fun.",Amy did not think Alaska would be and decided ...,Amy did not think Alaska would be fun but deci...,2,0.542582,0.457418,1
5,Ann loved cake.,Ann came inside and watched as cakes were bein...,The local baker let Ann come into the workshop...,She passed by a meat shop one day.,1,0.929146,0.070854,1
6,Anna was jogging with her phone in her pocket.,She was so relieved to see it wasn't cracked!,Anne dropped her phone.,Anna's phone fell out of her pocket into nearb...,1,0.456302,0.543698,2
7,Beth was a model in a fashion show at her school.,"Beth missed the show, and was unable to tell a...",Beth passed out from overeating.,Beth went out and broke her phone with no ride...,2,0.593895,0.406105,1
8,Beth's brother Jake came to her house to check...,Jake slept on the couch to keep her safe.,Jake couldn't find the source of the noise so ...,Jake was scared to go to sleep alone in the ho...,1,0.450498,0.549502,2
9,Bob decided to hike in the jungles of Indonesia.,Bob had also forgotten his shoes.,Bob stopped in the middle of the hike because ...,Bob stopped in the middle of his hike to tie h...,1,0.635864,0.364136,1


In [17]:
run.finish()

accuracy/test,▁
accuracy/train,▁▃▃▄▅▆▆▇█
accuracy/valid,▁▃▅██▇▇▇▅█
epoch,▁▁▂▂▃▃▃▃▄▄▅▅▆▆▆▆▇▇██
loss/test,▁
loss/train,█▇▇▆▅▅▃▂▁
loss/valid,▄▃▂▁▁▁▂▅█▁
trainer/global_step,▁▁▂▂▃▃▄▄▄▄▅▅▆▆▇▇████
accuracy/majority,0.50979
accuracy/test,0.51828
accuracy/train,0.85543
